# 🏏 Cricket Pitch & Weather Performance Benchmark: Starter Guide
### Analyzing How Air Density, Venue Altitude, Boundaries, and Pitch Soil Impact Cricket Outcomes

---

## 📌 Overview
This starter notebook demonstrates how to load, visualize, and model the **Cricket Pitch and Weather Performance Benchmark** dataset.

### Key Research Questions Covered:
1. **Aerodynamics & Six-Hitting**: How does moist air density ($\rho$) at varying altitudes affect six-hitting frequency?
2. **Pitch Soil Agronomy**: How do **Red Soil**, **Black Soil**, and **Drop-in wickets** compare in first-innings run rates and average scores?
3. **Dew & Innings Asymmetry**: Does a high dew risk index disproportionately hurt 2nd innings bowling economy for pace vs. spin bowlers?
4. **Granular Deliveries**: How to query and analyze 1.6M+ ball-by-ball deliveries using Apache Parquet.

In [ ]:
# 1. Imports and Environment Setup
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

# Auto-detect data path (works both on Kaggle and locally)
def find_data_path():
    kaggle_paths = glob.glob("/kaggle/input/**/matches_enriched.csv", recursive=True)
    if kaggle_paths:
        return os.path.dirname(kaggle_paths[0])
    local_paths = ["data/processed", ".", "kaggle_upload"]
    for p in local_paths:
        if os.path.exists(os.path.join(p, "matches_enriched.csv")):
            return p
    return "."

DATA_DIR = find_data_path()
print(f"Data directory detected at: {DATA_DIR}")
for f in os.listdir(DATA_DIR):
    if f.endswith((".csv", ".parquet")):
        size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024 * 1024)
        print(f"  - {f:<32} ({size_mb:.2f} MB)")

---
## 🏟️ 2. Loading Matches & Venue Characteristics

In [ ]:
df_matches = pd.read_csv(os.path.join(DATA_DIR, "matches_enriched.csv"))
df_venues = pd.read_csv(os.path.join(DATA_DIR, "venues_master.csv"))

print("Total matches loaded:", len(df_matches))
print("Total stadiums cataloged:", len(df_venues))

# Preview match sample
df_matches[["match_id", "season", "venue_name", "elevation_m", "air_density_kg_m3", "pitch_soil_type", "match_total_sixes", "innings1_runs"]].head()

---
## 🌬️ 3. Aerodynamics: Air Density & Elevation vs. Six-Hitting
In aerodynamics, drag force is given by $F_D = \frac{1}{2} \rho v^2 C_D A$. At high altitude venues (like Wanderers at 1,720m or Dharamsala at 1,457m), air density $\rho$ drops by up to 20%, resulting in lower aerodynamic drag and longer ball flight distances.

In [ ]:
# Filter for venues with at least 15 matches for reliable averages
venue_stats = df_matches.groupby("venue_name").filter(lambda g: len(g) >= 15).groupby("venue_name").agg(
    matches=("match_id", "count"),
    elevation_m=("elevation_m", "first"),
    air_density=("air_density_kg_m3", "mean"),
    avg_sixes=("match_total_sixes", "mean"),
    avg_boundary_m=("boundary_avg_m", "first")
).reset_index()

plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    venue_stats["air_density"],
    venue_stats["avg_sixes"],
    c=venue_stats["elevation_m"],
    s=venue_stats["matches"] * 2,
    cmap="viridis",
    edgecolor="black",
    alpha=0.85
)
cbar = plt.colorbar(scatter)
cbar.set_label("Elevation (meters above sea level)")

# Overlay regression trendline
sns.regplot(data=venue_stats, x="air_density", y="avg_sixes", scatter=False, color="crimson")
plt.title("Moist Air Density (rho) vs. Average Sixes per Match", fontsize=13, fontweight="bold")
plt.xlabel("Air Density (kg/m3) [Lower = Thinner Air at Altitude]")
plt.ylabel("Average Sixes Hit per Match")
plt.tight_layout()
plt.show()

---
## 🌿 4. Pitch Soil Profile vs. 1st Innings Scoring
Let's compare how pitch soil composition (Red Soil, Black Soil, Clay/Loam, Drop-in turf) shapes batting scoring potential across venues.

In [ ]:
# Filter soil categories with at least 30 matches
soil_counts = df_matches["pitch_soil_type"].value_counts()
top_soils = soil_counts[soil_counts >= 30].index
df_soils = df_matches[df_matches["pitch_soil_type"].isin(top_soils)]

soil_summary = df_soils.groupby("pitch_soil_type").agg(
    matches=("match_id", "count"),
    avg_1st_innings_runs=("innings1_runs", "mean"),
    avg_run_rate=("innings1_run_rate", "mean"),
    avg_sixes=("match_total_sixes", "mean")
).sort_values(by="avg_1st_innings_runs", ascending=False).round(2)

print("=== PITCH SOIL COMPOSITION BENCHMARK ===")
display(soil_summary)

plt.figure(figsize=(11, 5))
order = soil_summary.index
sns.boxplot(data=df_soils, x="innings1_runs", y="pitch_soil_type", order=order, palette="Spectral")
plt.title("1st Innings Total Score Distribution by Pitch Soil Profile", fontsize=13, fontweight="bold")
plt.xlabel("1st Innings Total Runs")
plt.ylabel("Pitch Soil Profile")
plt.tight_layout()
plt.show()

---
## 💧 5. Dew Index & Innings Asymmetry (Chasing Advantage)

In [ ]:
# Categorize Dew Levels
def get_dew_bracket(d):
    if d < 0.25:
        return "1. Low Dew Risk (< 0.25)"
    elif d < 0.60:
        return "2. Moderate Dew Risk (0.25 - 0.60)"
    else:
        return "3. High Dew Risk (>= 0.60)"

df_matches["dew_bracket"] = df_matches["dew_index"].apply(get_dew_bracket)
df_matches["chasing_won"] = (df_matches["match_winner"] == df_matches["innings2_team"]).astype(int)

dew_analysis = df_matches.groupby("dew_bracket").agg(
    matches=("match_id", "count"),
    chasing_win_pct=("chasing_won", lambda x: round(x.mean() * 100, 1)),
    avg_inn1_score=("innings1_runs", "mean"),
    avg_inn2_score=("innings2_runs", "mean"),
    avg_inn1_rr=("innings1_run_rate", "mean"),
    avg_inn2_rr=("innings2_run_rate", "mean")
).round(2)

print("=== DEW RISK BENCHMARK ===")
display(dew_analysis)

---
## ⚡ 6. Granular Ball-by-Ball Analysis with Parquet
Let's load the 1.6M+ delivery dataset to analyze **Pace vs. Spin bowling performance across match phases** (Powerplay, Middle, Death).

In [ ]:
parquet_file = os.path.join(DATA_DIR, "deliveries_enriched.parquet")

# Load key columns from Parquet efficiently
columns_to_load = ["phase", "bowler_type", "innings", "runs_total", "is_dot", "is_boundary_six", "is_wicket"]
table = pq.read_table(parquet_file, columns=columns_to_load)
df_deliv = table.to_pandas()

print("Total deliveries loaded from Parquet:", len(df_deliv))

# Phase Breakdown: Pace vs Spin Economy and Dot %
phase_stats = df_deliv.groupby(["phase", "bowler_type"]).agg(
    balls=("runs_total", "count"),
    total_runs=("runs_total", "sum"),
    wickets=("is_wicket", "sum"),
    dots=("is_dot", "sum")
).reset_index()

phase_stats["overs"] = phase_stats["balls"] / 6.0
phase_stats["economy_rate"] = (phase_stats["total_runs"] / phase_stats["overs"]).round(2)
phase_stats["dot_ball_pct"] = ((phase_stats["dots"] / phase_stats["balls"]) * 100).round(1)
phase_stats["strike_rate"] = (phase_stats["balls"] / phase_stats["wickets"]).round(1)

display(phase_stats[["phase", "bowler_type", "overs", "economy_rate", "dot_ball_pct", "strike_rate"]])

---
## 🤖 7. Baseline Predictive Model: Predicting 1st Innings Runs
Predicting total first-innings runs directly from pre-match environmental features (Elevation, Air Density, Boundary Distance, and Temperature).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Filter complete matches
model_df = df_matches[df_matches["innings1_overs"] >= 18].dropna(subset=[
    "elevation_m", "air_density_kg_m3", "boundary_avg_m", "temperature_c", "relative_humidity_pct", "innings1_runs"
]).copy()

features = ["elevation_m", "air_density_kg_m3", "boundary_avg_m", "temperature_c", "relative_humidity_pct"]
X = model_df[features]
y = model_df["innings1_runs"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
preds = rf.predict(X_test)

print("Baseline Random Forest MAE:", round(mean_absolute_error(y_test, preds), 2), "runs")

# Feature Importance
feat_imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
feat_imp.plot(kind="barh", color="teal")
plt.title("Condition Feature Importance for Expected 1st Innings Runs", fontsize=12, fontweight="bold")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()